<h3> Cleanning, classfying, and counting sentiment score distribution for the alternative headline, keywords, summary

In [ ]:
# Import necessary packages
import pandas as pd
import nltk
from nltk.corpus import stopwords
from nltk.tokenize.toktok import ToktokTokenizer
import re
from transformers import AutoConfig, AutoModelForSequenceClassification, AutoTokenizer 
import torch 
# Downloads list of stopwords from nltk package
nltk.download('stopwords')
# Initializes a stopwords list and tokenizer
stopWordlst = set(stopwords.words('english'))
textTokenizer = ToktokTokenizer()

# Loads data
path = ''
years = range(2015, 2020)
featuresdfs = [pd.read_csv(f'{path}{year}.csv') for year in years]
featuresdf = pd.concat(featuresdfs, ignore_index=True) # Shape [17452, 5]

# Extracts the date column
datedf = featuresdf.iloc[:, 0:1]

# Concatenates date with headline, keywords, and sum
headlinedf = pd.concat([datedf, featuresdf.iloc[:, 2:3]], axis=1) # Shape [17452, 2]
keywordsdf = pd.concat([datedf, featuresdf.iloc[:, 3:4]], axis=1) # Shape [17452, 2]
sumdf = pd.concat([datedf, featuresdf.iloc[:, 4:5]], axis=1) # Shape [17452, 2]

# Defines a function to clean the text
def preprocess_text(text):
    text = text.lower() # Lowercases
    text = re.sub(r'\[.*?\]', '', text) # Removes square brackets and content
    text = re.sub(r"[,’\.!?:\-;()–']", '', text) # Removes punctuation
    text = re.sub(r'\s+', ' ', text).strip() # Removes extra spaces
    tokens = textTokenizer.tokenize(text) # Tokenizes
    filtered_tokens = [token for token in tokens if token not in stopWordlst] # Removes stopwords
    return ' '.join(filtered_tokens)

# Applies the function to the 'Generated Headline', 'Keywords', 'Generated Summary' columns
headlinedf['Cleaned Generated Headline'] = headlinedf['Generated Headline'].astype(str).apply(preprocess_text)
keywordsdf['Cleaned Keywords'] = keywordsdf['Keywords'].astype(str).apply(preprocess_text)
sumdf['Cleaned Generated Summary'] = sumdf['Generated Summary'].astype(str).apply(preprocess_text)

# Retrieves the two configured files for the CrudeBert
configPath = './crude_bert_config.json' 
modelPath = './crude_bert_model.bin'
# Loads the configuration
config = AutoConfig.from_pretrained(configPath)
# Creates the model from the configuration
model = AutoModelForSequenceClassification.from_config(config)
# Loads the model's state dictionary
state_dict = torch.load(modelPath)
# Inspects keys, if "bert.embeddings.position_ids" is unexpected, remove or adjust it
state_dict.pop("bert.embeddings.position_ids", None)
# Loads the adjusted state dictionary into the model
model.load_state_dict(state_dict, strict=False) # Using strict=False to ignore non-critical mismatches
# Loads the tokenizer
BERTTokenizer = AutoTokenizer.from_pretrained('bert-base-uncased')

# Defines the classification function
def predictionSentiment(texts, model, tokenizer): 
    model.eval() 
    data = [] 
    totalTexts = len(texts)

    for i, text in enumerate(texts):
        print(f"Processing {i+1}/{totalTexts}...", end="\r")
        inputs = tokenizer(text, return_tensors="pt", padding=True, truncation=True, max_length=64) 
        
        with torch.no_grad(): 
            outputs = model(**inputs) 
            logits = outputs.logits 
            softmax_scores = torch.nn.functional.softmax(logits, dim=-1) 

            pred_label_id = torch.argmax(softmax_scores, dim=-1).item() 
            class_names = ['positive', 'negative', 'neutral'] 
            predicted_label = class_names[pred_label_id]

            # Extract and round probabilities
            softmax_probs = softmax_scores.tolist()[0]
            positive_prob = round(softmax_probs[0], 2)
            negative_prob = round(softmax_probs[1], 2)
            neutral_prob  = round(softmax_probs[2], 2)

            data.append([text, predicted_label, positive_prob, negative_prob, neutral_prob])

    df = pd.DataFrame(data, columns=["Headline", "Classification", "Positive", "Negative", "Neutral"])
    return df

outputFolder = 'Textual Features (GPT Features)'

# --- Process Cleaned Headline ---
headlineClassificationdf = predictionSentiment(headlinedf['Cleaned Generated Headline'].tolist(), model, BERTTokenizer)
headlineClassificationdf.insert(0, 'Date', headlinedf['Date'].values)
headlineClassificationdf.rename(columns={"Headline": "Generated Headline"}, inplace=True)
headlineClassificationdf.to_csv(f'{outputFolder}/Cleaned (Alternative Headline).csv', index=False)

# --- Process Cleaned Keywords ---
keywordClassificationdf = predictionSentiment(keywordsdf['Cleaned Keywords'].tolist(), model, BERTTokenizer)
keywordClassificationdf.insert(0, 'Date', keywordsdf['Date'].values)
keywordClassificationdf.rename(columns={"Headline": "Keywords"}, inplace=True)
keywordClassificationdf.to_csv(f'{outputFolder}/Cleaned (Keywords).csv', index=False)

# --- Process Cleaned Summary ---
summaryClassificationdf = predictionSentiment(sumdf['Cleaned Generated Summary'].tolist(), model, BERTTokenizer)
summaryClassificationdf.insert(0, 'Date', sumdf['Date'].values)
summaryClassificationdf.rename(columns={"Headline": "Generated Summary"}, inplace=True)
summaryClassificationdf.to_csv(f'{outputFolder}/Cleaned (Summary).csv', index=False)

<h3> Choosing one textual feature per day, seven textual features per week

In [ ]:
# Import necessary packages
import pandas as pd
from datetime import timedelta

# Loads the datasets
headlinedf = pd.read_csv('Textual Features (GPT Features)/Cleaned (Alternative Headline).csv')
keywordsdf = pd.read_csv('Textual Features (GPT Features)/Cleaned (Keywords).csv')
sumdf = pd.read_csv('Textual Features (GPT Features)/Cleaned (Summary).csv')
# Converts 'Date' column to datetime format
headlinedf['Date'] = pd.to_datetime(headlinedf['Date'], format='%Y%m%d', errors='coerce') # Shape [17452, 6]
keywordsdf['Date'] = pd.to_datetime(keywordsdf['Date'], format='%Y%m%d', errors='coerce') # Shape [17452, 6]
sumdf['Date'] = pd.to_datetime(sumdf['Date'], format='%Y%m%d', errors='coerce') # Shape [17452, 6]

# Defines the rolling 7-day grouping function
def createDateGroupColumn(df, start="2015-01-01", end="2019-12-31"):
    dateRange = pd.date_range(start=start, end=end)
    groupedRows = []
    for groupDate in dateRange:
        startDate = groupDate - timedelta(days=6)
        mask = (df['Date'] >= startDate) & (df['Date'] <= groupDate)
        group = df[mask].copy()
        group['Date Group'] = groupDate
        groupedRows.append(group)
    weeklyDf = pd.concat(groupedRows, ignore_index=True)
    cols = list(weeklyDf.columns)
    cols.insert(1, cols.pop(cols.index('Date Group')))
    return weeklyDf[cols]

# Applies the function to each DataFrame
headlineWeeklydf = createDateGroupColumn(headlinedf) # Shape [122078, 7]
keywordsWeeklydf = createDateGroupColumn(keywordsdf) # Shape [122078, 7]
summaryWeeklydf = createDateGroupColumn(sumdf) # Shape [122078, 7]

def selectWeeklySentiment(df):
    selectedRows = []
    for date_group, group_df in df.groupby('Date Group'):
        for date, date_df in group_df.groupby('Date'):
            classification_counts = date_df['Classification'].value_counts()
            max_count = classification_counts.max()
            top_classes = classification_counts[classification_counts == max_count].index.tolist()

            for classification in top_classes:
                class_df = date_df[date_df['Classification'] == classification]
                score_col = classification.capitalize()
                if score_col in class_df.columns:
                    top_row = class_df.loc[class_df[score_col].idxmax()]
                    selectedRows.append(top_row)
    return pd.DataFrame(selectedRows)

# Applies selection
headlineSelected = selectWeeklySentiment(headlineWeeklydf)
keywordsSelected = selectWeeklySentiment(keywordsWeeklydf)
summarySelected = selectWeeklySentiment(summaryWeeklydf)

outputFolder = 'Textual Features (GPT Features)'

headlineSelected.to_csv(f'{outputFolder}/Choosen (Alternative Headline).csv', index=False)
keywordsSelected.to_csv(f'{outputFolder}/Choosen (Keywords).csv', index=False)
summarySelected.to_csv(f'{outputFolder}/Choosen (Summary).csv', index=False)

<h3> Word embedding using FinBERT and sentence embedding using average pooling

In [ ]:
import pandas as pd
from transformers import AutoTokenizer, AutoModel
from tqdm import tqdm
import torch
import random
import numpy as np

# Setups for reproducibility
seed = 42
random.seed(seed)
np.random.seed(seed)
torch.manual_seed(seed)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Loads FinBERT
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModel.from_pretrained("ProsusAI/finbert")

# Defines a function to process and embed one dataset
def process_and_embed(file_path, text_column, group_column, output_csv_name, label_name):
    # Loads dataset
    df = pd.read_csv(file_path)
    # Combines text by Date Group
    combined_df = df.groupby(group_column)[text_column].apply(lambda x: ' '.join(x)).reset_index()
    combined_df.columns = ['Date', label_name]
    # Tokenizes
    texts = combined_df[label_name].tolist()
    encoding = tokenizer.batch_encode_plus(
        texts,
        padding=True,
        truncation=True,
        return_tensors='pt',
        add_special_tokens=True
    )
    input_ids = encoding['input_ids']
    attention_mask = encoding['attention_mask']
    # FinBERT forward pass (in batches)
    with torch.no_grad():
        outputs = []
        for i in tqdm(range(0, len(input_ids), 32), desc=f"Processing {label_name}", unit="batch"):
            batch_input_ids = input_ids[i:i+32]
            batch_attention_mask = attention_mask[i:i+32]
            batch_output = model(batch_input_ids, attention_mask=batch_attention_mask)
            outputs.append(batch_output.last_hidden_state)

    word_embeddings = torch.cat(outputs, dim=0) # Shape [1826, 143, 106, 420, 768]
    print(word_embeddings.shape)
    sentence_embeddings = word_embeddings.mean(dim=1) # Shape [1826, 768]
    print(sentence_embeddings.shape)

    # Creates DataFrame
    sentence_embeddings_np = sentence_embeddings.numpy()
    embeddings_df = pd.DataFrame(sentence_embeddings_np)
    embeddings_df.insert(0, "Date", combined_df["Date"])
    embeddings_df.insert(1, label_name, texts)

    # Saves to CSV
    embeddings_df.to_csv(output_csv_name, index=False)
    print(f"Saved: {output_csv_name}")

# Process: Generated Headline
process_and_embed(
    file_path='Textual Features (GPT Features)/Choosen (Alternative Headline).csv',
    text_column='Generated Headline',
    group_column='Date Group',
    output_csv_name='Textual Features (GPT Features)/Sentence Embedding (Alternative Headline).csv',
    label_name='Generated Headline'
)

# Process: Keywords
process_and_embed(
    file_path='Textual Features (GPT Features)/Choosen (Keywords).csv',
    text_column='Keywords',
    group_column='Date Group',
    output_csv_name='Textual Features (GPT Features)/Sentence Embedding (Keywords).csv',
    label_name='Keywords'
)

# Process: Generated Summary
process_and_embed(
    file_path='Textual Features (GPT Features)/Choosen (Summary).csv',
    text_column='Generated Summary',
    group_column='Date Group',
    output_csv_name='Textual Features (GPT Features)/Sentence Embedding (Summary).csv',
    label_name='Generated Summary'
)